In [1]:
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/sasha/miniconda3/envs/lima/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_PATH = Path("../data/qwen_train_data.jsonl")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 203.00it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [3]:
examples = []

with DATA_PATH.open("r", encoding="utf-8") as f:
    for _ in range(2):
        examples.append(json.loads(f.readline()))

In [4]:
examples = []

with DATA_PATH.open("r", encoding="utf-8") as f:
    for _ in range(2):
        examples.append(json.loads(f.readline()))

In [5]:
for i, example in enumerate(examples):
    print(f"\nExample {i + 1}")
    print(example["text"][:1000])


Example 1
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Can brain cells move? By movement I mean long distance migration (preferably within the brain only).<|im_end|>
<|im_start|>assistant
The question is relatively broad and one should take into account that the brain not only consists of neurons, but also glial cells (supportive cells) and pre-mitotic neuronal stem cells. Furthermore, as critical fellow-scientists have indicated, developmental stage is very important, as the developing embryonic brain is very different from the adult brain.
However, after sifting through various publications, the answer to the question is actually remarkably simple: Yes, brain cells migrate.
In  the adult brain glial cells migrate in the brain (Klämbt, 2009). Glial cells are involved in a myriad of functions, but a notable example of migrating glial cells are the oligodendrocytes that migrate relative long distances to find their t

In [6]:
texts = [example["text"] for example in examples]

In [8]:
batch = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
)

print(f"Input IDs: {tuple(batch['input_ids'].shape)}")
print(f"Attention Mask: {tuple(batch['attention_mask'].shape)}")

Input IDs: (2, 822)
Attention Mask: (2, 822)


In [9]:
labels = batch["input_ids"].clone()
labels[batch["attention_mask"] == 0] = -100

print(f"Labels shape : {tuple(labels.shape)}")

Labels shape : (2, 822)


In [10]:
with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=labels,
    )

print(f"Loss: {outputs.loss.item():.4f}")
print(f"Logits shape: {tuple(outputs.logits.shape)}")

Loss: 2.9959
Logits shape: (2, 822, 151936)


In [ ]:
shifted_logits = outputs.logits[:, :-1, :]     # removes last prediction
shifted_labels = labels[:, 1:]

print(f"Original logits: {tuple(outputs.logits.shape)}")
print(f"Shifted logits: {tuple(shifted_logits.shape)}")
print(f"Original labels: {tuple(labels.shape)}")
print(f"Shifted labels: {tuple(shifted_labels.shape)}")

Original logits: (2, 822, 151936)
Shifted logits: (2, 821, 151936)
Original labels: (2, 822)
Shifted labels: (2, 821)


In [13]:
flattened_logits = shifted_logits.reshape(-1, shifted_logits.size(-1))
flattened_labels = shifted_labels.reshape(-1)

print(f"Flattened logits: {tuple(flattened_logits.shape)}")
print(f"Flattened labels: {tuple(flattened_labels.shape)}")

Flattened logits: (1642, 151936)
Flattened labels: (1642,)


In [16]:
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)
manual_loss = loss_fn(flattened_logits, flattened_labels)

print(f"Model-computed loss: {outputs.loss.item():.6f}")
print(f"Manually computed loss: {manual_loss.item():.6f}")
print(f"Losses match: {torch.allclose(outputs.loss, manual_loss)}")     # |input - other| ≤ atol + rtol × |other|

Model-computed loss: 2.995932
Manually computed loss: 2.995932
Losses match: True


In [17]:
ignored_tokens = (flattened_labels == -100).sum().item()
used_tokens = (flattened_labels != -100).sum().item()

print(f"Tokens used in loss: {used_tokens}")
print(f"Tokens ignored: {ignored_tokens}")
print(f"Total positions: {flattened_labels.numel()}")

Tokens used in loss: 1231
Tokens ignored: 411
Total positions: 1642


In [18]:
row = 0

for pos in range(20, 40):
    current = tokenizer.decode([batch["input_ids"][row, pos].item()])
    target_id = shifted_labels[row, pos].item()
    target = "<ignored>" if target_id == -100 else tokenizer.decode([target_id])

    print(
        f"pos={pos:3} | "
        f"current={repr(current):15} | "
        f"target_next={repr(target)}"
    )

pos= 20 | current='\n'            | target_next='<|im_start|>'
pos= 21 | current='<|im_start|>'  | target_next='user'
pos= 22 | current='user'          | target_next='\n'
pos= 23 | current='\n'            | target_next='Can'
pos= 24 | current='Can'           | target_next=' brain'
pos= 25 | current=' brain'        | target_next=' cells'
pos= 26 | current=' cells'        | target_next=' move'
pos= 27 | current=' move'         | target_next='?'
pos= 28 | current='?'             | target_next=' By'
pos= 29 | current=' By'           | target_next=' movement'
pos= 30 | current=' movement'     | target_next=' I'
pos= 31 | current=' I'            | target_next=' mean'
pos= 32 | current=' mean'         | target_next=' long'
pos= 33 | current=' long'         | target_next=' distance'
pos= 34 | current=' distance'     | target_next=' migration'
pos= 35 | current=' migration'    | target_next=' ('
pos= 36 | current=' ('            | target_next='prefer'
pos= 37 | current='prefer'        | target_